| Threshold | # Chunks | Precision@10 |  Recall@10 |    MAP@100 |     MRR@10 |    NDCG@10 |
| --------- | -------: | -----------: | ---------: | ---------: | ---------: | ---------: |
| 0.7       |   371,351 |   **0.2227** |     0.5802 |     0.5506 | **0.9682** |     0.6571 |
| 0.6       |  235,396 |       0.1687 |     0.6731 |     0.6284 |     0.9526 |     0.7114 |
| 0.5       |  107,053 |       0.1182 |     0.8338 |     0.7760 |     0.9197 |     0.8216 |
| 0.4       |  107,053 |       0.1187 | **0.8362** | **0.7789** |     0.9229 | **0.8243** |
| 0.0       |  65,069 |       0.0862 |     0.8390 |     0.6786 |     0.6789 |     0.7158 |


In [1]:
import json
import pandas as pd

INPUT_JSONL = "chunks_embeddings_v1.jsonl"

corpus_rows = []
query_rows = []
qrels_rows = []

heading_to_qid = {}
query_counter = 0
corpus_counter = 0

In [2]:
with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)

        heading = item.get("heading")
        chunk = item.get("chunk")

        if not heading or not chunk:
            continue

        heading = str(heading).strip()
        chunk = str(chunk).strip()

        if not heading or not chunk:
            continue

        # Query ID
        if heading not in heading_to_qid:
            qid = f"q_{query_counter}"
            heading_to_qid[heading] = qid

            query_rows.append({
                "id": qid,
                "text": heading
            })

            query_counter += 1

        qid = heading_to_qid[heading]

        # Corpus ID
        cid = f"c_{corpus_counter}"
        corpus_counter += 1

        corpus_rows.append({
            "id": cid,
            "text": chunk
        })

        # Relevant mapping
        qrels_rows.append({
            "query_id": qid,
            "corpus_id": cid,
            "relevance": 1
        })

In [18]:
# Save
corpus_df = pd.DataFrame(corpus_rows)
queries_df = pd.DataFrame(query_rows)
qrels_df = pd.DataFrame(qrels_rows)

# corpus_df.to_parquet("corpus.parquet", index=False)
# queries_df.to_parquet("queries.parquet", index=False)
# qrels_df.to_parquet("data_ir.parquet", index=False)

print("Corpus :", len(corpus_df))
print("Queries:", len(queries_df))
print("Qrels  :", len(qrels_df))

Corpus : 107053
Queries: 59930
Qrels  : 107053


In [4]:
import random
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from datasets import load_dataset
import pandas as pd

# Load a model
from dotenv import load_dotenv
import os
load_dotenv()
hf_token = os.environ["HF_TOKEN"]
# model = SentenceTransformer(r"D:/HocTap/ChatBot/Legal_Embedding/768/binhdinh-embedding-e8/binhdinh-embedding/checkpoint-1696", token=hf_token, device="cuda")
# model = SentenceTransformer(r"D:\HocTap\ChatBot\Legal_Embedding\768-256\binhdinh-embedding-bestmodel-epoch6\kaggle\working\binhdinh-embedding-bestmodel-epoch6")
model = SentenceTransformer("bkai-foundation-models/vietnamese-bi-encoder", token=hf_token, device="cuda")



d:\HocTap\ChatBot\Medical_Chatbot\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Windows\AppData\Local\Temp\ipykernel_24948\3120190930.py:3: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import InformationRetrievalEvaluator
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9003.27it/s]


In [19]:
import random

# ===== 1. Chọn 1000 query =====
selected_queries = random.sample(
    queries_df["id"].tolist(),
    k=1000
)

queries_df = queries_df[queries_df["id"].isin(selected_queries)]

# ===== 2. Giữ lại qrels của các query này =====
qrels_df = qrels_df[qrels_df["query_id"].isin(selected_queries)]

# ===== 3. Lấy toàn bộ corpus liên quan =====
required_corpus_ids = set(qrels_df["corpus_id"].astype(str))

all_corpus_ids = corpus_df["id"].astype(str).tolist()

# ===== 4. Thêm corpus ngẫu nhiên cho đủ ~10k =====
remaining_ids = list(set(all_corpus_ids) - required_corpus_ids)

num_random = max(0, 10000 - len(required_corpus_ids))

required_corpus_ids |= set(
    random.sample(remaining_ids, min(num_random, len(remaining_ids)))
)

# ===== 5. Lọc corpus =====
corpus_df = corpus_df[
    corpus_df["id"].astype(str).isin(required_corpus_ids)
]

print("Corpus :", len(corpus_df))
print("Queries:", len(queries_df))
print("Qrels  :", len(qrels_df))

Corpus : 10000
Queries: 1000
Qrels  : 1656


In [20]:
# Convert the datasets to dictionaries
corpus = dict(zip(corpus_df["id"], corpus_df["text"]))  # Our corpus (cid => document)
queries = dict(zip(queries_df["id"], queries_df["text"]))  # Our queries (qid => question)
relevant_docs = {}  # Query ID to relevant documents (qid => set([relevant_cids])
for qid, corpus_ids in zip(qrels_df["query_id"], qrels_df["corpus_id"]):
    qid = str(qid)
    corpus_ids = str(corpus_ids)
    if qid not in relevant_docs:
        relevant_docs[qid] = set()
    relevant_docs[qid].add(corpus_ids)

In [21]:
# Given queries, a corpus and a mapping with relevant documents, the InformationRetrievalEvaluator computes different IR metrics.
ir_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="bkai-chunking-eval",
    batch_size=8,
    # truncate_dim=256
)

In [22]:
results = ir_evaluator(model)

In [23]:
from pprint import pprint
pprint(results)

{'bkai-chunking-eval_cosine_accuracy@1': 0.888,
 'bkai-chunking-eval_cosine_accuracy@10': 0.972,
 'bkai-chunking-eval_cosine_accuracy@3': 0.95,
 'bkai-chunking-eval_cosine_accuracy@5': 0.961,
 'bkai-chunking-eval_cosine_map@100': 0.7759960737397654,
 'bkai-chunking-eval_cosine_mrr@10': 0.9196523809523811,
 'bkai-chunking-eval_cosine_ndcg@10': 0.821626666155528,
 'bkai-chunking-eval_cosine_precision@1': 0.888,
 'bkai-chunking-eval_cosine_precision@10': 0.11820000000000001,
 'bkai-chunking-eval_cosine_precision@3': 0.3516666666666666,
 'bkai-chunking-eval_cosine_precision@5': 0.22420000000000004,
 'bkai-chunking-eval_cosine_recall@1': 0.6970806277056276,
 'bkai-chunking-eval_cosine_recall@10': 0.8338484307359307,
 'bkai-chunking-eval_cosine_recall@3': 0.7809495129870131,
 'bkai-chunking-eval_cosine_recall@5': 0.8084206168831168}
